## Minimal example of flow-based market coupling clearing

In [1]:
import pypsa
import fbmc  # registers pypsa.Network.fbmc accessor

from fbmc.settings import FBMCConfig

### Create a simple three-node network

In [2]:
nodal_net = pypsa.Network()
nodal_net.set_snapshots(['1', '2'])
nodal_net.add('Bus', ['A1', 'B1', 'B2'])
nodal_net.buses.loc[:, 'zone_name'] = ['A', 'B', 'B']
nodal_net.add('Line', 'B1-A1', bus0='B1', bus1='A1', x=1, s_nom=12)
nodal_net.add('Line', 'B1-B2', bus0='B1', bus1='B2', x=1, s_nom=12)
nodal_net.add('Line', 'A1-B2', bus0='A1', bus1='B2', x=1, s_nom=12)

nodal_net.add('Generator', 'gen_A1', bus='A1', p_nom=100, marginal_cost=400, carrier="Oil")
nodal_net.add('Generator', 'gen_B1', bus='B1', p_nom=100, marginal_cost=100, carrier="Wind")
nodal_net.add('Generator', 'gen_B2', bus='B2', p_nom=100, marginal_cost=200, carrier="CCGT")
nodal_net.add('Load', 'load_A1', bus='A1', p_set=[15, 15])


Index(['load_A1'], dtype='object')

In [3]:
zonal_net = nodal_net.fbmc.to_zonal(nodal_net.buses['zone_name'])

In [4]:
config = FBMCConfig.from_base_yaml("config/base_config.yaml")
print(config)

FBMCConfig:
cnec_setting: ALL
add_security_constraints: false
security_constraint_bodf_size_threshold: 0.2
security_constraint_bodf_columnwise_matrix_size_limit: 5000000
gsk_strategy: P_NOM
gsk_kwargs: null
reliability_margin_factor: 0.0
min_ram: 0.0
upper_ram_only_flag: false
base_case_strategy: zero_flows
marginal_cost_load_shedding: 5000.0
advanced_hybrid_coupling_flag: false
create_model_kwargs: {}
solver_kwargs:
  solver_name: gurobi
  OutputFlag: 0


In [5]:
zonal_net.fbmc.create_model(nodal_net, config=config)
zonal_net.model.solve(**config.solver_kwargs)
fbmc_result = zonal_net.fbmc.results()

INFO:root:Determined 1 sub-networks in the base case nodal network.
Index(['gen_A1', 'gen_B1', 'gen_B2'], dtype='object', name='Generator')
Index(['A', 'B'], dtype='object', name='Bus')
INFO:root:Created optimization model without meshed split.


Fixed load has coordinate Bus


INFO:linopy.model: Solve problem using Gurobi solver
INFO:linopy.model:Solver options:
 - OutputFlag: 0
INFO:linopy.io: Writing time: 0.12s


Set parameter Username


INFO:gurobipy:Set parameter Username


...

INFO:gurobipy:Read LP format model from file C:\Users\wouterko\AppData\Local\Temp\linopy-problem-7vium9hu.lp


Reading time = 0.00 seconds


INFO:gurobipy:Reading time = 0.00 seconds


obj: 26 rows, 10 columns, 34 nonzeros


INFO:gurobipy:obj: 26 rows, 10 columns, 34 nonzeros
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 10 primals, 26 duals
Objective: 3.00e+03
Solver model: available
Solver message: 2



In [6]:

print(fbmc_result.dispatch_results)
print(fbmc_result.net_positions)

DispatchResult object with attrs: 
  generators_p: (2, 3) snapshots x generators, 
  storage_units_p: (2, 0) snapshots x storage units, 
  links_p0: (2, 0) snapshots x links, 
  storage_levels: None snapshots x storage levels, 
  water_values: None snapshots x water values
<xarray.DataArray 'Zone-p' (snapshot: 2, Zone: 2)> Size: 32B
array([[-15.,  15.],
       [-15.,  15.]])
Coordinates:
  * snapshot  (snapshot) object 16B '1' '2'
  * Zone      (Zone) <U1 8B 'A' 'B'
